# 🎨 SD Image Generator
Stable Diffusion XL · ipywidgets UI · YouTube-ready sizes · Face mode

In [ ]:
import subprocess, sys, os

os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/working/hf_cache"

def install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

PACKAGES = [
    "diffusers",
    "accelerate",
    "transformers",
    "huggingface_hub",
    "ipywidgets",
    "insightface",
    "onnxruntime-gpu",
    "opencv-python-headless",
]
for pkg in PACKAGES:
    install(pkg)

import torch
import ipywidgets as widgets
from IPython.display import display, clear_output
import random, io, base64
from PIL import Image

print(f"✅ Ready | PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
STYLE_MODELS = {
    "Realism": "SG161222/RealVisXL_V4.0",
    "Anime": "cagliostrolab/animagine-xl-3.1",
    "Comics": "Lykon/dreamshaper-xl-1-0",
    "Illustration": "playgroundai/playground-v2.5-1024px-aesthetic",
    "Lineart": "stabilityai/stable-diffusion-xl-base-1.0",
}

SIZES = {
    "YouTube Video (1920x1080)": (1920, 1080),
    "YouTube Shorts (1080x1920)": (1080, 1920),
}

import gc
from diffusers import DiffusionPipeline, StableDiffusionXLPipeline
from huggingface_hub import file_exists

def release_pipeline_resources(*pipelines):
    for pipeline in pipelines:
        if pipeline is None:
            continue
        try:
            pipeline.unload_ip_adapter()
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

def refiner_supported_for_pipe(pipe):
    return isinstance(pipe, StableDiffusionXLPipeline)

def load_pipeline(model_id: str, load_refiner: bool = False):
    global refiner

    try:
        has_diffusers_format = file_exists(model_id, "model_index.json")
    except Exception:
        has_diffusers_format = False

    if not has_diffusers_format:
        raise ValueError(
            f"{model_id} does not expose model_index.json. Add an explicit checkpoint filename before using single-file loading."
        )

    try:
        pipe = DiffusionPipeline.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            use_safetensors=True,
            variant="fp16",
        )
    except Exception:
        pipe = DiffusionPipeline.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            use_safetensors=True,
        )

    pipe.enable_model_cpu_offload()
    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass

    if load_refiner:
        refiner = DiffusionPipeline.from_pretrained(
            "stabilityai/stable-diffusion-xl-refiner-1.0",
            text_encoder_2=pipe.text_encoder_2,
            vae=pipe.vae,
            torch_dtype=torch.float16,
            use_safetensors=True,
            variant="fp16",
        )
        refiner.enable_model_cpu_offload()
        try:
            refiner.enable_xformers_memory_efficient_attention()
        except Exception:
            pass
    else:
        refiner = None

    return pipe

pipe = None
refiner = None

style_radio = widgets.RadioButtons(
    options=list(STYLE_MODELS.keys()),
    description="Style:",
    layout=widgets.Layout(width="400px"),
)
load_btn = widgets.Button(
    description="Load model",
    button_style="primary",
    layout=widgets.Layout(width="200px"),
)
model_status = widgets.Label(value="No model loaded")

refiner_checkbox = widgets.Checkbox(
    value=False,
    description="Load refiner (better quality, slower)",
    layout=widgets.Layout(width="400px"),
)

def on_load(btn):
    global pipe, refiner

    model_status.value = f"Loading {style_radio.value}..."
    load_btn.disabled = True

    old_pipe, old_refiner = pipe, refiner
    pipe = None
    refiner = None
    release_pipeline_resources(old_pipe, old_refiner)

    try:
        pipe = load_pipeline(STYLE_MODELS[style_radio.value], load_refiner=refiner_checkbox.value)
        status = f"{style_radio.value} ready"
        if refiner_checkbox.value:
            if refiner_supported_for_pipe(pipe):
                status += " + refiner"
            else:
                status += " | refiner loaded but not supported for this style"
        model_status.value = status
    except Exception as e:
        model_status.value = f"Error: {e}"
        pipe = None
        refiner = None
    finally:
        load_btn.disabled = False

load_btn.on_click(on_load)
display(widgets.VBox([
    widgets.Label("Select style and load model:"),
    style_radio,
    refiner_checkbox,
    widgets.HBox([load_btn, model_status]),
]))

In [ ]:
from diffusers import StableDiffusionXLPipeline
from PIL import ImageOps

DEFAULT_NEGATIVE_PROMPT = (
    "deformed, ugly, blurry, low quality, worst quality, duplicate, multiple people, "
    "extra limbs, extra arms, extra hands, extra fingers, fused fingers, malformed hands, bad anatomy"
)

DEFAULT_FACE_ADAPTER_SCALE = 0.7
BASE_STEPS = 35
REFINER_STEPS = 40
HIGH_NOISE_FRACTION = 0.8

def image_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

def show_gallery(images, seeds_used):
    cards = []
    for i, (img, seed) in enumerate(zip(images, seeds_used)):
        b64 = image_to_b64(img)
        dl = widgets.HTML(
            f'<a download="img_{i+1}_seed{seed}.png" href="data:image/png;base64,{b64}">'
            f'<button>Download</button></a>'
        )
        thumb = widgets.Image(value=base64.b64decode(b64), format="png", width=320, height=180)
        cards.append(widgets.VBox([thumb, dl]))
    rows = [widgets.HBox(cards[i:i+4]) for i in range(0, len(cards), 4)]
    display(widgets.VBox(rows))

def extract_uploaded_bytes(upload_value):
    if not upload_value:
        return None
    if isinstance(upload_value, dict):
        first_item = next(iter(upload_value.values()))
        return first_item["content"]
    first_item = upload_value[0]
    if isinstance(first_item, dict):
        return first_item["content"]
    return first_item.content

def normalize_face_image(face_bytes, max_side=1024):
    face_img = Image.open(io.BytesIO(face_bytes))
    face_img = ImageOps.exif_transpose(face_img).convert("RGB")
    face_img.thumbnail((max_side, max_side))
    return face_img

def make_generator(seed):
    return torch.Generator(device="cuda").manual_seed(int(seed))

def build_generation_plan(count):
    if seed_mode_radio.value == "Random each time":
        seeds = [random.randint(0, 2**32 - 1) for _ in range(count)]
        return {
            "mode": "random",
            "items": [
                {
                    "display_seed": seed,
                    "generator": make_generator(seed),
                    "refiner_generator": make_generator(seed),
                }
                for seed in seeds
            ],
        }

    base_seed = max(0, int(seed_input.value))
    return {
        "mode": "fixed",
        "base_seed": base_seed,
        "count": count,
    }

def load_face_pipeline(pipe, face_bytes, adapter_scale):
    face_img = normalize_face_image(face_bytes)
    if not getattr(pipe, "_ip_adapter_loaded", False):
        pipe.load_ip_adapter(
            "h94/IP-Adapter",
            subfolder="sdxl_models",
            weight_name="ip-adapter-plus-face_sdxl_vit-h.safetensors",
        )
        pipe._ip_adapter_loaded = True
    pipe.set_ip_adapter_scale(adapter_scale)
    return pipe, face_img

def run_base_pipeline(pipe, prompt, negative_prompt, width, height, guidance, generator, refiner_generator, face_img=None):
    common_kwargs = {
        "prompt": prompt,
        "negative_prompt": negative_prompt,
        "width": width,
        "height": height,
        "generator": generator,
        "guidance_scale": guidance,
        "num_images_per_prompt": 1,
    }
    if face_img is not None:
        common_kwargs["ip_adapter_image"] = face_img

    if refiner is not None and isinstance(pipe, StableDiffusionXLPipeline):
        latents = pipe(
            num_inference_steps=REFINER_STEPS,
            denoising_end=HIGH_NOISE_FRACTION,
            output_type="latent",
            **common_kwargs,
        ).images
        return refiner(
            prompt=prompt,
            negative_prompt=negative_prompt,
            image=latents,
            generator=refiner_generator,
            num_inference_steps=REFINER_STEPS,
            denoising_start=HIGH_NOISE_FRACTION,
            guidance_scale=guidance,
        ).images[0]

    return pipe(
        num_inference_steps=BASE_STEPS,
        **common_kwargs,
    ).images[0]

def generate_batch(pipe, prompt, negative_prompt, generation_plan, width, height, guidance, face_img=None, refiner_warning=None):
    images = []
    display_seeds = []
    if generation_plan["mode"] == "fixed":
        base_seed = generation_plan["base_seed"]
        total = max(generation_plan["count"], 1)
        for index in range(1, total + 1):
            display_seed = base_seed
            display_seeds.append(display_seed)
            status_label.value = f"Generating {index}/{total} (base seed {display_seed})..."
            if refiner_warning:
                status_label.value += f" {refiner_warning}"
            progress_bar.value = 10 + int(80 * (index - 1) / total)
            image = run_base_pipeline(
                pipe,
                prompt,
                negative_prompt,
                width,
                height,
                guidance,
                generator=make_generator(generation_plan["base_seed"]),
                refiner_generator=make_generator(generation_plan["base_seed"]),
                face_img=face_img,
            )
            images.append(image)
            progress_bar.value = 10 + int(80 * index / total)
        return images, display_seeds

    items = generation_plan["items"]
    total = max(len(items), 1)
    for index, item in enumerate(items, start=1):
        display_seed = item["display_seed"]
        display_seeds.append(display_seed)
        status_label.value = f"Generating {index}/{total} (seed {display_seed})..."
        if refiner_warning:
            status_label.value += f" {refiner_warning}"
        progress_bar.value = 10 + int(80 * (index - 1) / total)
        image = run_base_pipeline(
            pipe,
            prompt,
            negative_prompt,
            width,
            height,
            guidance,
            generator=item["generator"],
            refiner_generator=item["refiner_generator"],
            face_img=face_img,
        )
        images.append(image)
        progress_bar.value = 10 + int(80 * index / total)
    return images, display_seeds

prompt_ta = widgets.Textarea(
    placeholder="Describe what you want to generate...",
    description="Prompt:",
    layout=widgets.Layout(width="600px", height="80px"),
)
neg_prompt_ta = widgets.Textarea(
    value=DEFAULT_NEGATIVE_PROMPT,
    description="Negative:",
    layout=widgets.Layout(width="600px", height="80px"),
)
count_slider = widgets.IntSlider(
    value=2, min=1, max=8, step=1,
    description="Count:",
    layout=widgets.Layout(width="400px"),
)
guidance_slider = widgets.FloatSlider(
    value=6.5, min=1.0, max=12.0, step=0.5,
    description="Guidance:",
    layout=widgets.Layout(width="400px"),
)
seed_input = widgets.IntText(
    value=42,
    description="Seed:",
    layout=widgets.Layout(width="200px"),
)
random_seed_btn = widgets.Button(
    description="Random seed",
    layout=widgets.Layout(width="160px"),
)
seed_mode_radio = widgets.RadioButtons(
    options=["Fixed", "Random each time"],
    value="Fixed",
    description="Seed mode:",
)
size_radio = widgets.RadioButtons(
    options=list(SIZES.keys()),
    description="Size:",
)
face_upload = widgets.FileUpload(
    accept="image/*",
    description="Face (opt.):",
    layout=widgets.Layout(width="300px"),
)
face_adapter_slider = widgets.FloatSlider(
    value=DEFAULT_FACE_ADAPTER_SCALE, min=0.0, max=1.0, step=0.05,
    description="Face strength:",
    layout=widgets.Layout(width="400px"),
)
generate_btn = widgets.Button(
    description="Generate",
    button_style="success",
    layout=widgets.Layout(width="200px", height="40px"),
)
progress_bar = widgets.IntProgress(
    value=0, min=0, max=100,
    description="Progress:",
    layout=widgets.Layout(width="400px"),
)
status_label = widgets.Label(value="Ready")
gallery_output = widgets.Output()

def on_random_seed(btn):
    seed_input.value = random.randint(0, 2**32 - 1)

random_seed_btn.on_click(on_random_seed)

def on_generate(btn):
    if pipe is None:
        status_label.value = "Load a model first (run the cell above)"
        return

    generate_btn.disabled = True
    progress_bar.value = 0
    status_label.value = "Preparing generation..."

    try:
        width, height = SIZES[size_radio.value]
        guidance = guidance_slider.value
        prompt = prompt_ta.value.strip()
        negative_prompt = neg_prompt_ta.value.strip() or DEFAULT_NEGATIVE_PROMPT
        generation_plan = build_generation_plan(count_slider.value)
        if generation_plan["mode"] == "fixed":
            seed_input.value = generation_plan["base_seed"]
        else:
            seed_input.value = generation_plan["items"][0]["display_seed"]

        refiner_warning = None
        if refiner is not None and not isinstance(pipe, StableDiffusionXLPipeline):
            refiner_warning = "Warning: refiner is inactive for this style."

        face_bytes = extract_uploaded_bytes(face_upload.value)
        face_img = None

        if face_bytes:
            _, face_img = load_face_pipeline(pipe, face_bytes, face_adapter_slider.value)
        elif getattr(pipe, "_ip_adapter_loaded", False):
            pipe.unload_ip_adapter()
            pipe._ip_adapter_loaded = False

        progress_bar.value = 10
        images, display_seeds = generate_batch(
            pipe,
            prompt,
            negative_prompt,
            generation_plan,
            width,
            height,
            guidance,
            face_img=face_img,
            refiner_warning=refiner_warning,
        )

        with gallery_output:
            clear_output(wait=True)
            show_gallery(images, display_seeds)

        progress_bar.value = 100
        if seed_mode_radio.value == "Fixed":
            status_label.value = f"Done: {len(images)} image(s) | seed: {display_seeds[0]}"
        else:
            status_label.value = f"Done: {len(images)} image(s) | seeds: {display_seeds[0]}..{display_seeds[-1]}"
        if refiner_warning:
            status_label.value += f" | {refiner_warning}"
    except Exception as e:
        status_label.value = f"Error: {e}"
    finally:
        generate_btn.disabled = False

generate_btn.on_click(on_generate)

display(widgets.VBox([
    prompt_ta,
    neg_prompt_ta,
    widgets.HBox([count_slider, guidance_slider]),
    widgets.HBox([seed_input, random_seed_btn]),
    seed_mode_radio,
    size_radio,
    face_upload,
    face_adapter_slider,
    generate_btn,
    widgets.HBox([progress_bar, status_label]),
]))
display(gallery_output)